In [27]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [28]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [29]:
import json


def generate_dataset():
  messages = []
  prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

  add_user_message(messages, prompt)
  add_assistant_message(messages, "```json")

  text = chat(messages, stop_sequences=["```"])

  return json.loads(text)

In [30]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
  json.dump(dataset, f, indent=2)

In [31]:
def run_prompt(test_case):
  """Merges the prompt and test case input, then returns the result"""
  prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

  messages = []
  add_user_message(messages, prompt)
  output = chat(messages)

  return output

In [32]:
def run_test_case(test_case):
  """Calls run_prompt, then grades the result"""
  output = run_prompt(test_case)
  
  # TODO - Grading
  score = 10

  return {
    "output": output,
    "test_case": test_case,
    "score": score
  }

In [33]:
def run_eval(dataset):
  """Loads the dataset and calls run test_case with each case"""
  results = []

  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  return results

In [ ]:
with open("dataset.json", "r") as f:
  dataset = json.load(f)

results = run_eval(dataset)

In [36]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a solution that handles multiple S3 URL formats:\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_s3_region(bucket_url: str) -> Optional[str]:\n    \"\"\"\n    Extract the AWS region from an S3 bucket URL.\n    \n    Supports multiple S3 URL formats:\n    - Virtual-hosted-style: https://bucket-name.s3.region.amazonaws.com\n    - Virtual-hosted-style (default region): https://bucket-name.s3.amazonaws.com\n    - Path-style: https://s3.region.amazonaws.com/bucket-name\n    - Path-style (default region): https://s3.amazonaws.com/bucket-name\n    \n    Args:\n        bucket_url: The S3 bucket URL\n        \n    Returns:\n        The AWS region code (e.g., 'us-west-2'), or None if not found\n    \"\"\"\n    # Pattern for virtual-hosted-style: bucket.s3.region.amazonaws.com\n    virtual_hosted_match = re.search(\n        r'\\.s3[.-]([a-z0-9\\-]+)\\.amazonaws\\.com',\n        bucket_url\n    )\n    if virtual